In [ ]:
!pip install datasets surprise pandas

In [ ]:
# --- 1. 라이브러리 임포트 및 데이터 로드 ---
import pandas as pd
import numpy as np
import pickle
from datasets import load_dataset

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

print("원본 데이터 로딩 중 (MichaelAHn/NVify_dataset)")
hf_dataset = load_dataset("MichaelAHn/NVify_dataset", split="train")
df = hf_dataset.to_pandas()

print(f"원본 데이터 로드 완료: {len(df)}개")
print(df.head())

In [ ]:
print("\n[자산 1] CF 모델 학습 시작 ---")

# 'rating' 컬럼이 0.01 ~ 1.0 범위이므로 rating_scale=(0, 1)로 설정
reader = Reader(rating_scale=(0, 1))
data = Dataset.load_from_df(df[['user_id', 'track_id', 'rating']], reader)

# API 서버에 쓸 최종 모델이므로, '전체 데이터'로 학습시킴
print("   - SVD 모델을 전체 데이터셋으로 학습 중... (시간이 걸릴 수 있습니다)")
trainset = data.build_full_trainset()
model_cf = SVD(n_factors=100, n_epochs=20, random_state=42, verbose=False)
model_cf.fit(trainset)

# [자산 1] 파일로 저장
with open('cf_model_final.pkl', 'wb') as f:
    pickle.dump(model_cf, f)

print("[자산 1] 'cf_model_final.pkl' 저장 완료 ---")

In [ ]:
# 트랙 메타 DB 구축
print("\n 트랙 메타 DB 구축 시작")

# 트랙별 인기도(평가 횟수) 계산
print(" - 트랙별 인기도(total_rating_count) 집계 중...")
rating_counts = df['track_id'].value_counts().reset_index()
rating_counts.columns = ['track_id', 'total_rating_count']

# 트랙별 V/A 정보 추출 (중복 제거)
print("   - 트랙별 V/A 정보 추출 중...")
track_info = df[['track_id', 'valence', 'energy']].drop_duplicates(subset=['track_id'])

# 인기도(1)와 V/A(2)를 병합
print("   - 인기도 + V/A 정보 병합 중...")
meta_df = pd.merge(track_info, rating_counts, on='track_id', how='left')
meta_df['total_rating_count'] = meta_df['total_rating_count'].fillna(0).astype(int)

# Pandas DataFrame을 최종 딕셔너리 형태로 변환
# (API 서버에서 빠르게 조회할 수 있도록 {track_id: {info}} 구조로 만듦)
track_meta_db = meta_df.set_index('track_id').to_dict('index')

# [자산 2] 파일로 저장
with open('track_meta_db.pkl', 'wb') as f:
    pickle.dump(track_meta_db, f)

print(f"[자산 2] 'track_meta_db.pkl' 저장 완료 (총 {len(track_meta_db)}개 트랙) ---")

In [ ]:
!pip install lightgbm scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import pickle
import lightgbm as lgb
from datasets import load_dataset
from sklearn.model_selection import train_test_split as sklearn_split
from surprise import SVD

In [ ]:
# [자산 1, 2] 로드 (방금 STEP 1에서 생성
print("[자산 1, 2] 로드 중 ---")
final_cf_model = pickle.load(open('cf_model_final.pkl', 'rb'))
track_meta_db = pickle.load(open('track_meta_db.pkl', 'rb'))
print("   - cf_model_final.pkl, track_meta_db.pkl 로드 완료")


# 학습 및 테스트 데이터셋 로드 ---
print("학습/테스트 데이터셋 로딩 중 ---")

# 학습 데이터 (Train)
hf_dataset_train = load_dataset("MichaelAHn/NVify_dataset", split="train")
df_train = hf_dataset_train.to_pandas()
print(f"   - 학습 데이터(NVify) 로드 완료: {len(df_train)}개")

# 테스트 데이터 (Test)
try:
    hf_dataset_test = load_dataset("Acervans/Lastfm-VADS", split="train")
    df_test = hf_dataset_test.to_pandas()
    print(f"   - 테스트 데이터(Lastfm-VADS) 로드 완료: {len(df_test)}개")

except Exception as e:
    print(f"   - Acervans/Lastfm-VADS 로드 실패: {e}")
    print("   - 테스트 데이터 없이, 학습 데이터를 80:20으로 분할하여 평가합니다.")
    df_train, df_test = sklearn_split(df_train, test_size=0.2, random_state=42)

In [ ]:
# LTR 피쳐 엔지니어링 ---
print("\n-LTR 피쳐 엔지니어링 시작 ")
USER_EMOTION_V = 0.5 # 학습용 가상 감정
USER_EMOTION_A = 0.5

def build_ltr_features(dataframe, df_name, cf_model, meta_db, user_emotion_v, user_emotion_a):

    print(f"--- {df_name} 피쳐 구축 시작 ---")

    # taste_score (자산 1 사용)
    print("   - 피쳐 1: 'taste_score' 계산 중...")
    dataframe['taste_score'] = dataframe.apply(
        lambda row: cf_model.predict(row['user_id'], row['track_id']).est,
        axis=1
    )

    # emotion_score, novelty_score
    print("   - 피쳐 2, 3: 'emotion_score', 'novelty_score' 계산 중...")

    def calculate_cb_scores(row, meta_db_ref, user_emov, user_emoa):
        track_id = row['track_id']
        meta = meta_db_ref.get(track_id)

        # DataFrame 자체의 valence, energy 사용
        track_valence = row['valence']
        track_energy = row['energy']

        # emotion_score (트랙 V/A와 가상 감정 V/A의 거리)
        distance = np.sqrt((user_emov - track_valence)**2 + (user_emoa - track_energy)**2)
        emotion_score = 1 / (1 + distance)

        # novelty_score (자산 2 사용)
        novelty_score = np.nan
        if meta and 'total_rating_count' in meta:
            novelty_score = 1 / (meta['total_rating_count'] + 1)

        # VADS 데이터셋에 'rating' 컬럼이 없다면 'label' 생성에서 오류 발생
        if 'rating' not in row:
             row['rating'] = 0.5 # 임시 처리 (실제 데이터에 맞게 수정 필요)

        return emotion_score, novelty_score, (row['rating'] >= 0.5)

    features_df = dataframe.apply(
        lambda row: calculate_cb_scores(row, meta_db, user_emotion_v, user_emotion_a),
        axis=1, result_type='expand'
    )
    dataframe['emotion_score'] = features_df[0]
    dataframe['novelty_score'] = features_df[1]

    # Label 및 Query 정리
    print("   - 피쳐 4, 5: 'Label', 'Query' 정리 중...")
    dataframe['label'] = features_df[2].astype(int)
    dataframe['query_id'] = dataframe['user_id'].astype('category').cat.codes

    # 최종 LTR 데이터셋 완성 (결측치 제거)
    ltr_dataframe = dataframe.dropna(subset=['taste_score', 'emotion_score', 'novelty_score', 'label'])
    print(f"{df_name} LTR 피쳐 구축 완료: {len(ltr_dataframe)}개")
    return ltr_dataframe

# 학습 데이터와 테스트 데이터에 각각 피쳐 구축 실행
ltr_train_df = build_ltr_features(df_train.copy(), "Train_DF", final_cf_model, track_meta_db, USER_EMOTION_V, USER_EMOTION_A)
ltr_test_df = build_ltr_features(df_test.copy(), "Test_DF", final_cf_model, track_meta_db, USER_EMOTION_V, USER_EMOTION_A)

In [ ]:
# LambdaMART 모델 학습 및 평가 ---
print("\nLambdaMART 모델 학습 및 평가 시작 ")

feature_columns = ['taste_score', 'emotion_score', 'novelty_score']

X_train = ltr_train_df[feature_columns]
y_train = ltr_train_df['label']
q_train = ltr_train_df.groupby('query_id').size().values

X_test = ltr_test_df[feature_columns]
y_test = ltr_test_df['label']
q_test = ltr_test_df.groupby('query_id').size().values

ranking_model = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    device='cpu',
    random_state=42,
    n_estimators=1000,
    learning_rate=0.05,
    verbose = -1
)

print("   - (평가용) 모델 학습 중... (Lastfm-VADS로 평가)")
if len(q_test) > 0 and len(q_train) > 0 and len(np.unique(q_test)) > 1:
    ranking_model.fit(
        X=X_train,
        y=y_train,
        group=q_train,
        eval_set=[(X_test, y_test)], # Last.fm-VADS로 평가
        eval_group=[q_test],
        eval_at=[5, 10], # NDCG@5, NDCG@10 계산
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)]
    )
else:
    print("   - 테스트 셋 쿼리 그룹이 부족하여 평가 없이 학습합니다.")
    ranking_model.fit(X=X_train, y=y_train, group=q_train)
print("   - (평가용) 모델 학습 완료.")


# [자산 3] 최종 모델 학습 및 저장
print("\n[자산 3] 최종 모델 (API 서빙용) 학습 시작 ---")
# API 서빙용 모델은 전체 학습 데이터(ltr_train_df)로 다시 학습
X_full = ltr_train_df[feature_columns]
y_full = ltr_train_df['label']
q_full = ltr_train_df.groupby('query_id').size().values

# 평가 모델에서 찾은 최적의 트리 개수를 사용합니다.
best_n_estimators = ranking_model.best_iteration_ if ranking_model.best_iteration_ else 500

final_ranking_model = lgb.LGBMRanker(
    objective='lambdarank',
    device='cpu',
    random_state=42,
    n_estimators=best_n_estimators,
    learning_rate=0.05,
    verbose = -1
)

final_ranking_model.fit(
    X=X_full,
    y=y_full,
    group=q_full
)

# [자산 3] 저장
with open('ranking_model_final.pkl', 'wb') as f:
    pickle.dump(final_ranking_model, f)
print("--- [자산 3] 'ranking_model_final.pkl' 저장 완료 ---")